#**Purpose**

This notebook runs inference on the fine tuned Flan-T5-Base model for question generation.

Model Trained on: **Kaggle**

## **Install the required modules**

In [34]:
!pip install -qU \
 transformers

**Restart the session after all the modules are installed.**

##**Import the required libraries**

In [35]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

##**Initialize the Model from Huggingface**

In [36]:
MODEL_REPO = "gaurav-dey/flan-t5-base-qg"
MAX_INPUT_LENGTH = 512                  # context+answer prompt length
MAX_TARGET_LENGTH = 96
NUM_BEAMS=4

In [37]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [38]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_REPO).to(device)

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.38k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/152 [00:00<?, ?B/s]

In [39]:
def generate_question(context, answer):
  prompt = (
      f"Target Answer: {answer}\n"
      f"Generate a question from the following context where the target answer is the correct answer. "
      f"Do not include phrases like 'According to the text' in the question and do not repeat the context in the question.\n"
      f"Context: {context}"
  )

  input_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LENGTH).input_ids.to(device)
  output_ids = model.generate(input_ids, max_length=MAX_TARGET_LENGTH, num_beams=NUM_BEAMS)
  question = tokenizer.decode(output_ids[0], skip_special_tokens=True)

  return question

In [41]:
context = "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower."
answer = "Gustave Eiffel"

question = generate_question(context, answer)
question

'Who is the Eiffel Tower named for?'

##**Huggingface Pipeline no longer supported for E-D Models in Transformers 5.X.X**

Note that we can not use a huggingface pipeline with encoder-decoder models like T5/Flan-T5/BART anymore as it is deprecated in transformers version 5 and above.

In [42]:
import transformers
transformers.__version__

'5.16.1'